# 🚀 DWT-MobileUNet: Huấn luyện Đa Vòng (Multi-Round Progressive Training), Hybrid Crop+Resize & Lượng tử hóa INT8 trên Google Colab
**Đề tài:** *Design of an Embedded System for Image Forgery Detection and Localization using Multiresolution Analysis and Deep Learning*

---
### 🎯 Kiến trúc & Chiến lược Huấn luyện Đã Tối ưu hóa Thực nghiệm:
1. **Kiến trúc `DWTMobileUNet`:** Tích hợp **2-Level 2D Haar DWT trực tiếp trên GPU** + nhánh nối tắt độ phân giải gốc (`192x192 Full-Res Edge Skip`) + `MobileNetV3-Small (width=0.5)` (`< 0.5 MB` ở dạng INT8).
2. **Huấn luyện Lai Đa Tỷ Lệ (`Hybrid Crop + Resize` — `--native_crop_prob`):**
   - Kết hợp **Resize toàn cảnh** (học ngữ cảnh ánh sáng/bố cục cho chế độ `fast`) và **Cắt trực tiếp 1:1 căn chỉnh lưới JPEG $8 \times 8$** (`x0 % 8 == 0, y0 % 8 == 0`, tâm xoay quanh vùng giả mạo cho chế độ `tile`).
3. **Hợp nhất Ngữ cảnh Toàn cục & Cục bộ (`Global-Local Context Fusion`) khi Suy luận `tile`:**
   - Kết hợp bản đồ toàn cảnh $M_{global}$ (35%) với bản đồ ghép cửa sổ trượt Gaussian Blending $M_{local}$ (65%), vừa giữ sắc nét vi mô từng pixel, vừa chống báo giả ở các góc ảnh rời rạc.
4. **3 Vòng Huấn luyện nối tiếp (Lưu trữ an toàn trên Google Drive):**
   - **🟢 Round 1 (`lr = 1e-3`, `native_crop_prob = 0.35`, 15 Epochs):** Học đặc trưng toàn cục + sóng con Haar DWT từ đầu (`--from-scratch`).
   - **🟡 Round 2 (`lr = 4e-4`, `native_crop_prob = 0.50`, 15 Epochs):** Tự động nạp `best_model.pth` từ Round 1, cân bằng 50/50 giữa Crop 1:1 và Resize để đẩy cao `Pixel-F1` và kéo `Au-FP` về `< 1%`.
   - **🔴 Round 3 (`lr = 1.5e-4`, `native_crop_prob = 0.50`, 15 Epochs):** Tự động nạp `best_model.pth` từ Round 2, hội tụ sâu ở vùng cực tiểu phẳng trước khi nén sang `INT8`.

### Bước 1: Kiểm tra GPU CUDA & Mount Google Drive

In [ ]:
!nvidia-smi

from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/DATN_DWT_MobileUNet'
DRIVE_CKPT_DIR = f'{DRIVE_PROJECT_DIR}/checkpoints'
DRIVE_DATA_DIR = f'{DRIVE_PROJECT_DIR}/dataset/CASIA2'

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'✅ Thư mục lưu Checkpoint bền vững trên Drive: {DRIVE_CKPT_DIR}')

### Bước 2: Clone / Pull mã nguồn mới nhất từ GitHub & Cài đặt thư viện

In [ ]:
GITHUB_REPO_URL = 'https://github.com/DinhKhanh27/forensics_system.git'
REPO_DIR = '/content/DATN_VER2'

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git reset --hard origin/main

%cd {REPO_DIR}
!pip install -q -r requirements_colab.txt
print('✅ Đã đồng bộ mã nguồn mới nhất và cài đặt thư viện!')

### Bước 3: Đồng bộ Dataset CASIA v2 sang ổ đĩa cục bộ `/content/dataset/CASIA2` của Colab
> Copy dữ liệu từ Google Drive sang ổ đĩa NVMe cục bộ của máy ảo Colab giúp tốc độ đọc ảnh mỗi Epoch nhanh gấp 8–10 lần.

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image

LOCAL_DATA_DIR = '/content/dataset/CASIA2'

if os.path.exists(DRIVE_DATA_DIR) and not os.path.exists(LOCAL_DATA_DIR):
    print(f'📂 Đang copy dataset từ {DRIVE_DATA_DIR} sang {LOCAL_DATA_DIR}...')
    os.makedirs('/content/dataset', exist_ok=True)
    !cp -r "{DRIVE_DATA_DIR}" /content/dataset/
    print('✅ Đã copy xong dataset sang ổ cứng cục bộ Colab!')
elif os.path.exists(LOCAL_DATA_DIR):
    print(f'✅ Dataset đã sẵn sàng tại {LOCAL_DATA_DIR}')
else:
    print('⚠️ Không tìm thấy DRIVE_DATA_DIR, tạo dataset mẫu để kiểm thử nhanh...')
    for sub in ['Au', 'Tp', 'Gt_mask']:
        Path(f'{LOCAL_DATA_DIR}/{sub}').mkdir(parents=True, exist_ok=True)
    for i in range(20):
        au = np.random.randint(40, 210, (256, 384, 3), dtype=np.uint8)
        Image.fromarray(au).save(f'{LOCAL_DATA_DIR}/Au/Au_sample_{i:03d}.jpg', quality=92)
        tp = np.random.randint(40, 210, (256, 384, 3), dtype=np.uint8)
        gt = np.zeros((256, 384), dtype=np.uint8)
        gt[64:168, 96:224] = 255
        tp[64:168, 96:224] = np.clip(tp[64:168, 96:224] + 55, 0, 255)
        Image.fromarray(tp).save(f'{LOCAL_DATA_DIR}/Tp/Tp_sample_{i:03d}.jpg', quality=75)
        Image.fromarray(gt).save(f'{LOCAL_DATA_DIR}/Gt_mask/Tp_sample_{i:03d}_gt.png')

### Bước 4: Kiểm thử nhanh `--dry-run` (3 giây)
Xác minh kiến trúc `DWTMobileUNet` (2-Level Haar DWT + Full-Res Edge Skip), hàm Loss đa nhiệm và CUDA AMP trước khi bước vào 3 vòng huấn luyện.

In [ ]:
!python train.py --dry-run --save_dir "{DRIVE_CKPT_DIR}" --amp

---
## 🔥 GIAI ĐOẠN HUẤN LUYỆN ĐA VÒNG (MULTI-ROUND PROGRESSIVE TRAINING)

### 🟢 Vòng 1 (Round 1): Khởi tạo Nền tảng Đa phân giải (`lr = 0.001`, `native_crop_prob = 0.35`, 15 Epochs)

In [ ]:
!python train.py \
    --data_dir "{LOCAL_DATA_DIR}" \
    --save_dir "{DRIVE_CKPT_DIR}" \
    --round_id 1 \
    --epochs 15 \
    --batch_size 16 \
    --lr 0.001 \
    --pos_pixel_weight 3.0 \
    --native_crop_prob 0.35 \
    --from-scratch \
    --amp

### 🟡 Vòng 2 (Round 2): Tăng Recall Vùng Giả mạo & Tinh chỉnh Biên (`lr = 0.0005`, `pos_pixel_weight = 5.0`, `focal_alpha = 0.55`, 40 Epochs)
Dựa trên kết quả Vòng 1 (`Au-FP` rất thấp ~`1.18%` nhưng `Pixel-F1` còn thấp `0.1101`), Vòng 2 tự động nạp `best_model.pth` từ Vòng 1, tăng `pos_pixel_weight` lên `5.0`, `focal_alpha` lên `0.55`, dùng `warmup_epochs = 3` + `grad_clip = 1.0`, `batch_size = 32` và `native_crop_prob = 0.45`.

In [ ]:
!python train.py \
    --data_dir "{LOCAL_DATA_DIR}" \
    --save_dir "{DRIVE_CKPT_DIR}" \
    --round_id 2 \
    --epochs 40 \
    --batch_size 32 \
    --lr 0.0005 \
    --warmup_epochs 3 \
    --pos_pixel_weight 5.0 \
    --focal_alpha 0.55 \
    --native_crop_prob 0.45 \
    --grad_clip 1.0 \
    --auto_resume \
    --amp

### 🔴 Vòng 3 (Round 3): Hội tụ Sâu & Chuẩn bị Lượng tử hóa INT8 (`lr = 0.00015`, `native_crop_prob = 0.50`, 15 Epochs)
Tự động nạp `best_model.pth` tốt nhất từ Vòng 2 và tinh chỉnh nhẹ ở `lr = 1.5e-4` để đạt `Pixel-F1` và `IoU` tối đa.

In [ ]:
!python train.py \
    --data_dir "{LOCAL_DATA_DIR}" \
    --save_dir "{DRIVE_CKPT_DIR}" \
    --round_id 3 \
    --epochs 15 \
    --batch_size 16 \
    --lr 0.00015 \
    --pos_pixel_weight 2.0 \
    --native_crop_prob 0.50 \
    --amp

---
### Bước 5: Vẽ Biểu đồ Hội tụ qua các Vòng (Multi-Round Training Curves)

In [ ]:
import json
import matplotlib.pyplot as plt

hist_file = f'{DRIVE_CKPT_DIR}/training_history.json'
if os.path.exists(hist_file):
    with open(hist_file, 'r') as f:
        hist = json.load(f)
    steps = list(range(1, len(hist) + 1))
    losses = [h['train_loss'] for h in hist]
    accs = [h['val_acc'] * 100 for h in hist]
    f1s = [h['val_pixel_f1'] * 100 for h in hist]
    au_fps = [h.get('au_fp_rate', 0.0) * 100 for h in hist]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(steps, losses, marker='o', color='#ef4444')
    axes[0].set_title('Train Multi-Task Loss')
    axes[0].set_xlabel('Total Epochs across Rounds')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(steps, accs, label='Val Cls Acc (%)', color='#3b82f6')
    axes[1].plot(steps, f1s, label='Val Pixel-F1 (%)', color='#10b981')
    axes[1].set_title('Validation Accuracy & Pixel-F1 (%)')
    axes[1].set_xlabel('Total Epochs across Rounds')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(steps, au_fps, color='#f59e0b', marker='s')
    axes[2].set_title('Authentic False-Positive Pixel Rate (%) ↓')
    axes[2].set_xlabel('Total Epochs across Rounds')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('Chưa có file training_history.json')

### Bước 6: Xuất ONNX & Lượng tử hóa INT8 (`< 2 MB`) cho Raspberry Pi 3
Đóng gói `torch.sigmoid` trực tiếp vào trong đồ thị `model_int8.onnx` để xác suất đầu ra luôn chuẩn xác trong `[0, 1]`.

In [ ]:
!python export_quantize.py \
    --checkpoint "{DRIVE_CKPT_DIR}/best_model.pth" \
    --output_dir "{DRIVE_CKPT_DIR}"

!cp -f "{DRIVE_CKPT_DIR}/model_int8.onnx" ./checkpoints/model_int8.onnx
!cp -f "{DRIVE_CKPT_DIR}/best_model.pth" ./checkpoints/best_model.pth
!ls -lh "{DRIVE_CKPT_DIR}"

### Bước 7: Kiểm chứng Thực nghiệm Trực quan (So sánh cả Chế độ `fast` và `tile` trên Ảnh Thật `Au` & Ảnh Giả `Tp`)
Chạy trực tiếp `EmbeddedForensicPredictor` với mô hình `model_int8.onnx` vừa xuất để kiểm chứng hiệu quả của **Global-Local Context Fusion**.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from src.dataset import scan_casia_forensic_dir
from rpi_inference import EmbeddedForensicPredictor

predictor = EmbeddedForensicPredictor(model_path=f'{DRIVE_CKPT_DIR}/model_int8.onnx', threshold=0.5)
samples, _ = scan_casia_forensic_dir(LOCAL_DATA_DIR)

au_item = next(s for s in reversed(samples) if s['label'] == 0.0)
tp_item = next(s for s in reversed(samples) if s['label'] == 1.0 and s['mask_path'] is not None)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row, (title_prefix, item) in enumerate([('ẢNH THẬT (Au)', au_item), ('ẢNH GIẢ (Tp)', tp_item)]):
    rgb = np.array(Image.open(item['img_path']).convert('RGB'))
    if item['mask_path']:
        gt_mask = np.array(Image.open(item['mask_path']).convert('L'))
    else:
        gt_mask = np.zeros(rgb.shape[:2], dtype=np.uint8)

    res = predictor.predict_image(rgb, mode='tile', threshold=0.5)

    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f"{title_prefix}: Ảnh Gốc ({res['resolution']})")
    axes[row, 0].axis('off')

    axes[row, 1].imshow(gt_mask, cmap='gray', vmin=0, vmax=255)
    axes[row, 1].set_title('Ground-Truth Mask')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(res['heatmap_img'])
    axes[row, 2].set_title(f"Heatmap (Mode: {res['mode_used']}, {res['num_patches']} patches)")
    axes[row, 2].axis('off')

    axes[row, 3].imshow(res['annotated_img'])
    axes[row, 3].set_title(f"{res['label']}\nP_fake = {res['p_fake']*100:.1f}% | {res['latency_ms']:.1f}ms")
    axes[row, 3].axis('off')

plt.tight_layout()
plt.show()

### Bước 8: Khởi chạy Giao diện Demo Gradio trực tiếp trên Google Colab (`share=True`)

In [ ]:
!python app_colab.py --model_path "{DRIVE_CKPT_DIR}/model_int8.onnx" --share